# Phoenix14T MSKA++ Pretrain

Stability-hardened Kaggle notebook for skeleton-only continuous sign language recognition. This MSKA++ version fixes spatial attention normalization and adds a stronger temporal encoder. Set `RUN_MODE = "smoke"` only for a quick path check.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import json
import math
import os
# Helps reduce CUDA memory fragmentation on Kaggle T4/P100 notebooks.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import gc
import random
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("Torch CUDA runtime:", torch.version.cuda)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.10.0+cu128
Torch CUDA runtime: 12.8
Device: cuda
GPU: Tesla T4


In [2]:
# Run mode:
#   full : full training run.
#   smoke: small subset for path/model sanity check if you need to re-check paths.
RUN_MODE = "full"

# MSKA++ is not checkpoint-compatible with the old MSKA notebook because the spatial
# attention and temporal encoder are changed. Train this notebook from scratch on PHOENIX14T.

EXPERIMENT_NAME = "phoenix14t_mska_plus_pretrain"
MODEL_NAME = "mska"
STAGE = "pretrain"
DATASET_NAME = "PHOENIX14T"

DATA_DIR = Path(r"/kaggle/input/datasets/thn1208/skeleton/skeleton")
META_PATH = DATA_DIR / "dataset_meta.json"
OUTPUT_DIR = Path("/kaggle/working") / EXPERIMENT_NAME
PRETRAIN_CKPT_PATH = Path(r"")
RESUME_CKPT_PATH = Path(r"/kaggle/input/datasets/thn1208/phase3/phoenix14t_mska_plus_pretrain/last_model.pt")

CONFIG = {
    "seed": 491,
    "batch_size": 2,
    "eval_batch_size": 1,
    "accum_steps": 8,
    "epochs": 180,
    "lr": 0.00008,
    "weight_decay": 0.0003,
    "warmup_epochs": 8,
    "patience": 25,
    "grad_clip": 0.5,
    "num_workers": 0,
    "hidden": 320,
    "dropout": 0.25,
    "adaptive_gcn": False,
    "aux_weight": 0.3,
    "lambda_kl": 0.0,
    "kl_temp": 1.0,
    "use_motion": True,
    "use_conf_gate": True,
    "num_blocks": 4,
    "num_heads": 8,
    "temporal_blocks": 4,
    "temporal_kernel_sizes": (3, 5, 7),
    "temporal_rnn_layers": 2,
    "stream_weight": 0.20,
    "stream_ramp_epochs": 60,
    "distill_warmup": 9999,
    "distill_temp": 2.0,
    "distill_max_weight": 0.0,
    "distill_step": 0.0,
    "root_normalize": True,
    "freeze_backbone_epochs": 0,
    "time_stretch_prob": 0.45,
    "time_stretch_range": (0.85, 1.15),
    "spatial_jitter_std": 0.005,
    "confidence_dropout_prob": 0.08,
    "confidence_dropout_ratio": 0.05,
    "enable_horizontal_flip": False,
    "horizontal_flip_prob": 0.0,
    "beam_width": 10,
    "beam_topk": 30,
    "blank_penalty": 0.35,
    "length_bonus": 0.01,
    "run_beam_in_smoke": False,
    "vocab_source": "all",
    "strict_oov_check": True,
    "smoke_train_samples": 32,
    "smoke_eval_samples": 8,
    "smoke_epochs": 2,
    "pin_memory": False,
    "empty_cache_each_epoch": True,
    "empty_cache_during_eval": True,
    "resume_training": True,
    "resume_load_optimizer": True,
    "amp_enabled": False,
    "skip_nonfinite_batches": True,
    "max_bad_batches_per_epoch": 30,
}

if RUN_MODE == "smoke":
    CONFIG["epochs"] = CONFIG["smoke_epochs"]
    CONFIG["patience"] = CONFIG["smoke_epochs"]
    CONFIG["num_workers"] = 0

print("Experiment:", EXPERIMENT_NAME)
print("Data dir:", DATA_DIR)
print("Metadata:", META_PATH)
print("Output dir:", OUTPUT_DIR)
print("Run mode:", RUN_MODE)
print("Resume checkpoint:", RESUME_CKPT_PATH if str(RESUME_CKPT_PATH) else "none")


Experiment: phoenix14t_mska_plus_pretrain
Data dir: /kaggle/input/datasets/thn1208/skeleton/skeleton
Metadata: /kaggle/input/datasets/thn1208/skeleton/skeleton/dataset_meta.json
Output dir: /kaggle/working/phoenix14t_mska_plus_pretrain
Run mode: full
Resume checkpoint: /kaggle/input/datasets/thn1208/phase3/phoenix14t_mska_plus_pretrain/last_model.pt


## Dataset and vocabulary

Vocabulary construction is controlled by `CONFIG["vocab_source"]`. VieCSL uses train-only vocabulary with strict OOV checking; Phoenix14T pretraining uses all official annotations because Phoenix dev/test contain a small number of glosses that do not appear in train.


In [3]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_metadata(meta_path: Path):
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    if not isinstance(meta, list):
        raise ValueError("Metadata must be a list of sample dictionaries.")
    required = {"video", "split", "gloss"}
    missing = [i for i, item in enumerate(meta) if not required.issubset(item)]
    if missing:
        raise ValueError(f"Metadata records missing keys at indices: {missing[:10]}")
    return meta


def build_vocab(meta, source: str = "train"):
    counter = Counter()
    for item in meta:
        if source == "all" or item["split"] == "train":
            counter.update(item["gloss"].split())
    if not counter:
        raise ValueError(f"No tokens found while building vocabulary from source={source!r}.")
    tokens = sorted(counter.keys(), key=lambda w: (-counter[w], w))
    vocab = {"<blank>": 0}
    for idx, token in enumerate(tokens, 1):
        vocab[token] = idx
    inv_vocab = {idx: token for token, idx in vocab.items()}
    return vocab, inv_vocab, counter


def collect_oov(meta, vocab):
    oov_by_split = defaultdict(set)
    for item in meta:
        for token in item["gloss"].split():
            if token not in vocab:
                oov_by_split[item["split"]].add(token)
    return {k: sorted(v) for k, v in oov_by_split.items()}


def assert_no_oov(meta, vocab):
    oov_by_split = collect_oov(meta, vocab)
    if oov_by_split:
        print("OOV tokens found:", oov_by_split)
        raise ValueError("Dev/test contains OOV tokens. Fix split or vocabulary first.")


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def save_csv(rows, path: Path, fieldnames=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        keys = set()
        for row in rows:
            keys.update(row.keys())
        fieldnames = sorted(keys)
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def resolve_npy_path(data_dir: Path, item: dict):
    rel = item.get("npy_path")
    if rel:
        path = data_dir / rel
        if path.exists():
            return path
    return data_dir / item["split"] / f"{item['video']}.npy"


class SkeletonDataset(Dataset):
    def __init__(self, meta, vocab, split: str, augment: bool):
        self.items = [item for item in meta if item["split"] == split]
        self.vocab = vocab
        self.split = split
        self.augment = augment

    def __len__(self):
        return len(self.items)

    def encode(self, gloss: str):
        return [self.vocab[token] for token in gloss.split()]

    @staticmethod
    def compute_motion(sk: np.ndarray):
        xy = sk[:, :, :2]
        dx_prev = np.concatenate([np.zeros_like(xy[:1]), xy[1:] - xy[:-1]], axis=0)
        dx_next = np.concatenate([xy[1:] - xy[:-1], np.zeros_like(xy[:1])], axis=0)
        motion = np.concatenate([dx_prev, dx_next], axis=-1)
        return np.clip(motion, -1.0, 1.0).astype(np.float32)

    @staticmethod
    def time_resample(arr: np.ndarray, new_len: int):
        if len(arr) == new_len:
            return arr
        idx = np.linspace(0, len(arr) - 1, new_len).astype(np.int64)
        return arr[idx]

    def augment_skeleton(self, sk: np.ndarray):
        if random.random() < CONFIG["time_stretch_prob"]:
            new_len = max(8, int(len(sk) * random.uniform(*CONFIG["time_stretch_range"])))
            sk = self.time_resample(sk, new_len)
        if CONFIG["spatial_jitter_std"] > 0:
            sk[:, :, :2] += np.random.randn(*sk[:, :, :2].shape).astype(np.float32) * CONFIG["spatial_jitter_std"]
        if random.random() < CONFIG["confidence_dropout_prob"]:
            drop = np.random.rand(sk.shape[0], sk.shape[1], 1) < CONFIG["confidence_dropout_ratio"]
            sk[:, :, 2:3] = np.where(drop, 0.0, sk[:, :, 2:3])
        if CONFIG["enable_horizontal_flip"] and random.random() < CONFIG["horizontal_flip_prob"]:
            sk[:, :, 0] = -sk[:, :, 0]
            sk_new = sk.copy()
            sk_new[:, 33:54, :] = sk[:, 54:75, :]
            sk_new[:, 54:75, :] = sk[:, 33:54, :]
            sk = sk_new
        return sk

    def __getitem__(self, idx):
        item = self.items[idx]
        path = resolve_npy_path(DATA_DIR, item)
        sk = np.load(path, allow_pickle=False).astype(np.float32)
        if sk.ndim != 3 or sk.shape[1] != 86 or sk.shape[2] != 3:
            raise ValueError(f"Expected skeleton shape (T, 86, 3), got {sk.shape} for {path}")
        sk = np.nan_to_num(sk, nan=0.0, posinf=0.0, neginf=0.0)
        if CONFIG["root_normalize"]:
            root_xy = sk[:, 0:1, :2].copy()
            sk[:, :, :2] = sk[:, :, :2] - root_xy
        if self.augment:
            sk = self.augment_skeleton(sk)

        motion = None
        if MODEL_NAME == "pipeline":
            motion = self.compute_motion(sk)
        elif CONFIG["use_motion"]:
            motion_xy = self.compute_motion(sk)[:, :, :2]
            sk = np.concatenate([sk, motion_xy], axis=-1).astype(np.float32)

        y = torch.tensor(self.encode(item["gloss"]), dtype=torch.long)
        sample = {
            "sk": torch.from_numpy(sk.astype(np.float32)),
            "mo": None if motion is None else torch.from_numpy(motion),
            "y": y,
            "input_len": int(sk.shape[0]),
            "target_len": int(len(y)),
            "item": item,
        }
        return sample


def collate_fn(batch):
    sk = pad_sequence([b["sk"] for b in batch], batch_first=True, padding_value=0.0)
    if batch[0]["mo"] is None:
        mo = None
    else:
        mo = pad_sequence([b["mo"] for b in batch], batch_first=True, padding_value=0.0)
    y = torch.cat([b["y"] for b in batch], dim=0)
    input_lens = torch.tensor([b["input_len"] for b in batch], dtype=torch.long)
    target_lens = torch.tensor([b["target_len"] for b in batch], dtype=torch.long)
    items = [b["item"] for b in batch]
    return sk, mo, y, input_lens, target_lens, items


set_seed(CONFIG["seed"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

meta = load_metadata(META_PATH)
vocab, inv_vocab, vocab_counter = build_vocab(meta, source=CONFIG["vocab_source"])
oov_report = collect_oov(meta, vocab)
if CONFIG["strict_oov_check"]:
    assert_no_oov(meta, vocab)
elif oov_report:
    print("OOV tokens are present but allowed by config:", oov_report)

split_counts = Counter(item["split"] for item in meta)
print("Split counts:", dict(split_counts))
print("Vocabulary size:", len(vocab), "(including blank)")
print("Vocabulary source:", CONFIG["vocab_source"])
print("Vocabulary token count:", sum(vocab_counter.values()))
if oov_report:
    print("OOV report:", oov_report)

save_json(vocab, OUTPUT_DIR / "vocab.json")
save_json(CONFIG, OUTPUT_DIR / "config.json")
save_json(oov_report, OUTPUT_DIR / "oov_report.json")

train_ds = SkeletonDataset(meta, vocab, "train", augment=True)
dev_ds = SkeletonDataset(meta, vocab, "dev", augment=False)
test_ds = SkeletonDataset(meta, vocab, "test", augment=False)

if RUN_MODE == "smoke":
    train_ds = Subset(train_ds, range(min(CONFIG["smoke_train_samples"], len(train_ds))))
    dev_ds = Subset(dev_ds, range(min(CONFIG["smoke_eval_samples"], len(dev_ds))))
    test_ds = Subset(test_ds, range(min(CONFIG["smoke_eval_samples"], len(test_ds))))

loader_base_kwargs = dict(
    collate_fn=collate_fn,
    num_workers=CONFIG["num_workers"],
    pin_memory=bool(torch.cuda.is_available() and CONFIG.get("pin_memory", False)),
)
if CONFIG["num_workers"] > 0:
    loader_base_kwargs["persistent_workers"] = False
    loader_base_kwargs["prefetch_factor"] = 2

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    drop_last=True,
    **loader_base_kwargs,
)
dev_loader = DataLoader(
    dev_ds,
    batch_size=CONFIG.get("eval_batch_size", CONFIG["batch_size"]),
    shuffle=False,
    drop_last=False,
    **loader_base_kwargs,
)
test_loader = DataLoader(
    test_ds,
    batch_size=CONFIG.get("eval_batch_size", CONFIG["batch_size"]),
    shuffle=False,
    drop_last=False,
    **loader_base_kwargs,
)

print(f"Train={len(train_ds)} Dev={len(dev_ds)} Test={len(test_ds)}")


Split counts: {'train': 7096, 'dev': 519, 'test': 642}
Vocabulary size: 1116 (including blank)
Vocabulary source: all
Vocabulary token count: 63259
Train=7096 Dev=519 Test=642


## Model definition


In [4]:
MSKA_STREAMS = {
    "body": [0, 11, 12, 13, 14, 15, 16, 23, 24],
    "lhand": list(range(33, 54)),
    "rhand": list(range(54, 75)),
    "face": list(range(75, 86)),
}
MSKA_STREAM_NAMES = ["body", "lhand", "rhand", "face", "fuse"]
INPUT_CHANNELS = 5 if CONFIG["use_motion"] else 3


class SpatialAttentionBlock(nn.Module):
    """Per-frame keypoint attention with proper softmax normalization and learnable spatial bias."""

    def __init__(self, d_model: int, n_points: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.q = nn.Linear(d_model, d_model, bias=False)
        self.k = nn.Linear(d_model, d_model, bias=False)
        self.v = nn.Linear(d_model, d_model, bias=False)
        self.o = nn.Linear(d_model, d_model)
        self.spatial_bias = nn.Parameter(torch.zeros(n_heads, n_points, n_points))
        self.attn_drop = nn.Dropout(CONFIG["dropout"] * 0.5)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(CONFIG["dropout"]),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(CONFIG["dropout"] * 0.5),
        )

    def forward(self, x):
        # x: (B, T, N, D)
        B, T, N, D = x.shape

        def heads(z):
            return z.view(B, T, N, self.n_heads, self.d_head).permute(0, 1, 3, 2, 4).contiguous()

        q = heads(self.q(x))
        k = heads(self.k(x))
        v = heads(self.v(x))
        with torch.amp.autocast("cuda", enabled=False):
            scores = torch.matmul(q.float(), k.float().transpose(-2, -1)) / math.sqrt(self.d_head)
            scores = scores + self.spatial_bias.float().unsqueeze(0).unsqueeze(0)
            attn = torch.softmax(scores, dim=-1)
            attn = self.attn_drop(attn)
            out = torch.matmul(attn, v.float())
        out = out.permute(0, 1, 3, 2, 4).contiguous().view(B, T, N, D)
        x = self.norm1(x + self.o(out.to(x.dtype)))
        x = self.norm2(x + self.ffn(x))
        return x


class MSKAStreamEncoder(nn.Module):
    """Part-aware keypoint encoder with confidence gating and learned attentive pooling."""

    def __init__(self, n_points: int):
        super().__init__()
        hidden = CONFIG["hidden"]
        self.input_norm = nn.LayerNorm(INPUT_CHANNELS)
        self.proj = nn.Linear(INPUT_CHANNELS, hidden)
        self.point_embed = nn.Parameter(torch.zeros(1, 1, n_points, hidden))
        self.conf_alpha = nn.Parameter(torch.tensor(0.5))
        self.blocks = nn.ModuleList([
            SpatialAttentionBlock(hidden, n_points, CONFIG["num_heads"])
            for _ in range(CONFIG["num_blocks"])
        ])
        self.pool_score = nn.Linear(hidden, 1)
        self.out_norm = nn.LayerNorm(hidden)
        nn.init.trunc_normal_(self.point_embed, std=0.02)

    def forward(self, x):
        # x: (B, T, N, C). Channel 2 is MediaPipe confidence/visibility when available.
        confidence = x[..., 2:3].clamp(0.0, 1.0) if x.shape[-1] >= 3 else None
        if CONFIG["use_conf_gate"] and confidence is not None:
            x = x * (1.0 + torch.sigmoid(self.conf_alpha) * confidence)
        h = self.proj(self.input_norm(x)) + self.point_embed
        for block in self.blocks:
            h = block(h)
        pool_logits = self.pool_score(h).squeeze(-1)
        if confidence is not None:
            pool_logits = pool_logits + torch.log(confidence.squeeze(-1).clamp_min(1e-4))
        weights = torch.softmax(pool_logits, dim=-1).unsqueeze(-1)
        return self.out_norm((h * weights).sum(dim=2))


class MultiScaleTemporalBlock(nn.Module):
    """Residual temporal block using parallel depthwise kernels for short and long movements."""

    def __init__(self, d_model: int, kernel_sizes):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(d_model, d_model, kernel_size=k, padding=k // 2, groups=d_model, bias=False),
                nn.BatchNorm1d(d_model),
                nn.GELU(),
            )
            for k in kernel_sizes
        ])
        self.mix = nn.Sequential(
            nn.Conv1d(d_model * len(kernel_sizes), d_model, kernel_size=1, bias=False),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(CONFIG["dropout"]),
        )
        self.ffn = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(CONFIG["dropout"]),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(CONFIG["dropout"] * 0.5),
        )

    def forward(self, x):
        # x: (B, T, D)
        z = self.norm(x).transpose(1, 2)
        z = torch.cat([branch(z) for branch in self.branches], dim=1)
        z = self.mix(z).transpose(1, 2)
        x = x + z
        x = x + self.ffn(x)
        return x


class StrongTemporalEncoder(nn.Module):
    """Multi-scale TCN plus BiGRU temporal encoder. Downsamples time once for CTC stability."""

    def __init__(self, d_model: int):
        super().__init__()
        self.blocks = nn.ModuleList([
            MultiScaleTemporalBlock(d_model, CONFIG["temporal_kernel_sizes"])
            for _ in range(CONFIG["temporal_blocks"])
        ])
        self.downsample = nn.MaxPool1d(kernel_size=2, stride=2, ceil_mode=True)
        self.rnn = nn.GRU(
            input_size=d_model,
            hidden_size=d_model // 2,
            num_layers=CONFIG["temporal_rnn_layers"],
            batch_first=True,
            bidirectional=True,
            dropout=CONFIG["dropout"] if CONFIG["temporal_rnn_layers"] > 1 else 0.0,
        )
        self.out = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(CONFIG["dropout"]),
        )

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        x = self.downsample(x.transpose(1, 2)).transpose(1, 2)
        self.rnn.flatten_parameters()
        x, _ = self.rnn(x)
        return self.out(x)


class AuxTemporalHead(nn.Module):
    """Light auxiliary CTC head for stream supervision."""

    def __init__(self, in_dim: int, vocab_size: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_dim, in_dim, kernel_size=5, padding=2, groups=in_dim, bias=False),
            nn.BatchNorm1d(in_dim),
            nn.GELU(),
            nn.Conv1d(in_dim, in_dim, kernel_size=1, bias=False),
            nn.BatchNorm1d(in_dim),
            nn.GELU(),
            nn.MaxPool1d(kernel_size=2, stride=2, ceil_mode=True),
            nn.Dropout(CONFIG["dropout"]),
        )
        self.cls = nn.Linear(in_dim, vocab_size)

    def forward(self, x):
        x = self.net(x.transpose(1, 2)).transpose(1, 2)
        return self.cls(x)


class FuseCTCHead(nn.Module):
    def __init__(self, in_dim: int, vocab_size: int):
        super().__init__()
        self.temporal = StrongTemporalEncoder(in_dim)
        self.cls = nn.Linear(in_dim, vocab_size)

    def forward(self, x):
        return self.cls(self.temporal(x))


class MSKACSLRModel(nn.Module):
    """MSKA++: normalized spatial keypoint attention + confidence pooling + multi-scale temporal fusion."""

    def __init__(self, vocab_size: int):
        super().__init__()
        hidden = CONFIG["hidden"]
        self.enc = nn.ModuleDict({name: MSKAStreamEncoder(len(indices)) for name, indices in MSKA_STREAMS.items()})
        self.stream_gate = nn.Sequential(
            nn.LayerNorm(hidden * 4),
            nn.Linear(hidden * 4, hidden),
            nn.GELU(),
            nn.Dropout(CONFIG["dropout"]),
            nn.Linear(hidden, 4),
        )
        self.fuse_proj = nn.Sequential(
            nn.LayerNorm(hidden * 4),
            nn.Linear(hidden * 4, hidden),
            nn.GELU(),
            nn.Dropout(CONFIG["dropout"]),
        )
        self.aux_heads = nn.ModuleDict({name: AuxTemporalHead(hidden, vocab_size) for name in MSKA_STREAMS})
        self.fuse_head = FuseCTCHead(hidden, vocab_size)

    def forward(self, sk):
        features = {name: self.enc[name](sk[:, :, indices, :]) for name, indices in MSKA_STREAMS.items()}
        ordered = [features[name] for name in MSKA_STREAMS]
        concat = torch.cat(ordered, dim=-1)
        gates = torch.softmax(self.stream_gate(concat), dim=-1)
        gated = torch.cat([feat * gates[:, :, i:i + 1] for i, feat in enumerate(ordered)], dim=-1)
        fused = self.fuse_proj(gated)
        aux = [self.aux_heads[name](features[name]) for name in MSKA_STREAMS]
        return tuple(aux + [self.fuse_head(fused)])


def create_model(vocab_size: int):
    return MSKACSLRModel(vocab_size)


## Training, decoding, and evaluation utilities

The notebook saves local artifacts for the report: curves, WER breakdowns, prediction CSV files, and final metrics JSON.


In [5]:
def pooled_lengths(input_lens: torch.Tensor, output_time: int):
    lens = (input_lens + 1) // 2
    return lens.clamp(min=1, max=output_time).cpu()


def ctc_loss_for_logits(logits, targets, input_lens, target_lens, ctc_fn):
    feat_lens = pooled_lengths(input_lens, logits.shape[1])
    log_probs = logits.log_softmax(-1).permute(1, 0, 2)
    return ctc_fn(log_probs, targets, feat_lens, target_lens.cpu())


def symmetric_kl(a, b, valid_lens, temperature: float):
    log_a = F.log_softmax(a / temperature, dim=-1)
    log_b = F.log_softmax(b / temperature, dim=-1)
    prob_a = log_a.exp()
    prob_b = log_b.exp()
    kl_ab = F.kl_div(log_a, prob_b.detach(), reduction="none").sum(-1)
    kl_ba = F.kl_div(log_b, prob_a.detach(), reduction="none").sum(-1)
    T = a.shape[1]
    mask = torch.arange(T, device=a.device)[None, :] < valid_lens.to(a.device)[:, None]
    return ((kl_ab + kl_ba) * 0.5 * mask).sum() / mask.sum().clamp(min=1)


def compute_training_loss(model, batch, ctc_fn, epoch: int):
    sk, mo, y, input_lens, target_lens, _ = batch
    sk = sk.to(DEVICE, non_blocking=True)
    y = y.to(DEVICE, non_blocking=True)
    input_lens_gpu = input_lens.to(DEVICE, non_blocking=True)
    if MODEL_NAME == "pipeline":
        mo = mo.to(DEVICE, non_blocking=True)
        aux, pri = model(sk, mo)
        pri_loss = ctc_loss_for_logits(pri, y, input_lens, target_lens, ctc_fn)
        aux_loss = ctc_loss_for_logits(aux, y, input_lens, target_lens, ctc_fn)
        loss = pri_loss + CONFIG["aux_weight"] * aux_loss
        if CONFIG["lambda_kl"] > 0:
            loss = loss + CONFIG["lambda_kl"] * symmetric_kl(aux, pri, pooled_lengths(input_lens_gpu, pri.shape[1]), CONFIG["kl_temp"])
        return loss, pri

    outputs = model(sk)
    fuse = outputs[-1]
    fuse_loss = ctc_loss_for_logits(fuse, y, input_lens, target_lens, ctc_fn)
    stream_loss = sum(ctc_loss_for_logits(outputs[i], y, input_lens, target_lens, ctc_fn) for i in range(4)) / 4.0
    gamma = min(CONFIG["stream_weight"], CONFIG["stream_weight"] * (epoch + 1) / max(1, CONFIG["stream_ramp_epochs"]))
    loss = fuse_loss + gamma * stream_loss
    if epoch >= CONFIG["distill_warmup"]:
        temp = CONFIG["distill_temp"]
        alpha = min(CONFIG["distill_max_weight"], (epoch - CONFIG["distill_warmup"] + 1) * CONFIG["distill_step"])
        with torch.no_grad():
            target = F.softmax(fuse.detach() / temp, dim=-1)
        distill = 0.0
        for i in range(4):
            distill = distill + F.kl_div(F.log_softmax(outputs[i] / temp, dim=-1), target, reduction="batchmean")
        loss = loss + alpha * (temp ** 2) * distill / 4.0
    return loss, fuse


def forward_logits(model, batch, head_index=-1):
    sk, mo, _, input_lens, _, _ = batch
    sk = sk.to(DEVICE, non_blocking=True)
    if MODEL_NAME == "pipeline":
        mo = mo.to(DEVICE, non_blocking=True)
        _, logits = model(sk, mo)
        return logits, input_lens
    outputs = model(sk)
    return outputs[head_index], input_lens


def edit_ops(ref_tokens, hyp_tokens):
    m, n = len(ref_tokens), len(hyp_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    back = [[None] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        dp[i][0] = i
        back[i][0] = "D"
    for j in range(1, n + 1):
        dp[0][j] = j
        back[0][j] = "I"
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i - 1] == hyp_tokens[j - 1]:
                choices = [(dp[i - 1][j - 1], "M")]
            else:
                choices = [(dp[i - 1][j - 1] + 1, "S")]
            choices.extend([(dp[i - 1][j] + 1, "D"), (dp[i][j - 1] + 1, "I")])
            dp[i][j], back[i][j] = min(choices, key=lambda x: x[0])
    i, j = m, n
    ops = []
    while i > 0 or j > 0:
        op = back[i][j]
        if op == "M":
            ops.append(("M", ref_tokens[i - 1], hyp_tokens[j - 1]))
            i -= 1
            j -= 1
        elif op == "S":
            ops.append(("S", ref_tokens[i - 1], hyp_tokens[j - 1]))
            i -= 1
            j -= 1
        elif op == "D":
            ops.append(("D", ref_tokens[i - 1], ""))
            i -= 1
        else:
            ops.append(("I", "", hyp_tokens[j - 1]))
            j -= 1
    ops.reverse()
    s = sum(1 for op, _, _ in ops if op == "S")
    d = sum(1 for op, _, _ in ops if op == "D")
    ins = sum(1 for op, _, _ in ops if op == "I")
    return {"S": s, "D": d, "I": ins, "N": max(1, m), "ops": ops}


def greedy_decode(log_probs, feat_lens, inv_vocab):
    best = log_probs.argmax(dim=-1).cpu()
    decoded = []
    for i in range(best.shape[0]):
        seq = best[i, : int(feat_lens[i])].tolist()
        out, prev = [], None
        for idx in seq:
            if idx != prev and idx != 0:
                out.append(inv_vocab.get(idx, ""))
            prev = idx
        decoded.append(" ".join(out))
    return decoded


def beam_decode_one(log_probs, inv_vocab, beam_size=5, topk=25, blank_penalty=0.0, length_bonus=0.0):
    arr = log_probs.detach().cpu().numpy().copy()
    if blank_penalty:
        arr[:, 0] -= float(blank_penalty)
    beams = {(): (0.0, -np.inf)}
    vocab_size = arr.shape[1]
    topk = min(topk, vocab_size)
    for t in range(arr.shape[0]):
        ids = np.argpartition(arr[t], -topk)[-topk:]
        if 0 not in ids:
            ids = np.concatenate([[0], ids])
        next_beams = defaultdict(lambda: [-np.inf, -np.inf])
        for prefix, (pb, pnb) in beams.items():
            total = np.logaddexp(pb, pnb)
            next_beams[prefix][0] = np.logaddexp(next_beams[prefix][0], total + arr[t, 0])
            for c in ids:
                c = int(c)
                if c == 0:
                    continue
                p = arr[t, c] + float(length_bonus)
                new_prefix = prefix + (c,)
                if prefix and c == prefix[-1]:
                    next_beams[new_prefix][1] = np.logaddexp(next_beams[new_prefix][1], pb + p)
                    next_beams[prefix][1] = np.logaddexp(next_beams[prefix][1], pnb + p)
                else:
                    next_beams[new_prefix][1] = np.logaddexp(next_beams[new_prefix][1], total + p)
        beams = dict(sorted(next_beams.items(), key=lambda kv: np.logaddexp(kv[1][0], kv[1][1]), reverse=True)[:beam_size])
    best = max(beams, key=lambda k: np.logaddexp(beams[k][0], beams[k][1])) if beams else ()
    return " ".join(inv_vocab.get(int(c), "") for c in best)


def decode_batch(log_probs, feat_lens, inv_vocab, method: str):
    if method == "greedy":
        return greedy_decode(log_probs, feat_lens, inv_vocab)
    return [
        beam_decode_one(
            log_probs[i, : int(feat_lens[i])],
            inv_vocab,
            CONFIG["beam_width"],
            CONFIG["beam_topk"],
            CONFIG.get("blank_penalty", 0.0),
            CONFIG.get("length_bonus", 0.0),
        )
        for i in range(log_probs.shape[0])
    ]


@torch.no_grad()
def evaluate(model, loader, inv_vocab, split_name: str, decode_method="greedy", head_index=-1):
    model.eval()
    rows = []
    for batch in tqdm(loader, desc=f"eval {split_name} {decode_method}", leave=False):
        logits, input_lens = forward_logits(model, batch, head_index=head_index)
        log_probs = logits.log_softmax(-1)
        feat_lens = pooled_lengths(input_lens, logits.shape[1])
        preds = decode_batch(log_probs, feat_lens, inv_vocab, decode_method)
        items = batch[-1]
        del logits, log_probs
        if torch.cuda.is_available() and CONFIG.get("empty_cache_during_eval", False):
            torch.cuda.empty_cache()
        for item, pred in zip(items, preds):
            ref = item["gloss"]
            stats = edit_ops(ref.split(), pred.split())
            wer = (stats["S"] + stats["D"] + stats["I"]) / stats["N"]
            rows.append({
                "video": item["video"],
                "split": item["split"],
                "reference": ref,
                "prediction": pred,
                "wer": wer,
                "S": stats["S"],
                "D": stats["D"],
                "I": stats["I"],
                "N": stats["N"],
                "ref_len": len(ref.split()),
                "frames": item.get("frames", ""),
                "lh_detect_rate": item.get("lh_detect_rate", ""),
                "rh_detect_rate": item.get("rh_detect_rate", ""),
                "pose_detect_rate": item.get("pose_detect_rate", ""),
            })
    totals = Counter()
    for row in rows:
        totals["S"] += int(row["S"])
        totals["D"] += int(row["D"])
        totals["I"] += int(row["I"])
        totals["N"] += int(row["N"])
    wer = (totals["S"] + totals["D"] + totals["I"]) / max(1, totals["N"])
    if torch.cuda.is_available() and CONFIG.get("empty_cache_during_eval", False):
        torch.cuda.empty_cache()
    metrics = {
        "split": split_name,
        "decode": decode_method,
        "wer": wer,
        "substitution_rate": totals["S"] / max(1, totals["N"]),
        "deletion_rate": totals["D"] / max(1, totals["N"]),
        "insertion_rate": totals["I"] / max(1, totals["N"]),
        "S": int(totals["S"]),
        "D": int(totals["D"]),
        "I": int(totals["I"]),
        "N": int(totals["N"]),
        "samples": len(rows),
    }
    return metrics, rows


def make_optimizer(model):
    return torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )


def make_scheduler(optimizer, total_epochs):
    warmup = CONFIG["warmup_epochs"]
    def lr_lambda(epoch):
        if warmup > 0 and epoch < warmup:
            return (epoch + 1) / warmup
        progress = (epoch - warmup) / max(1, total_epochs - warmup)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def set_backbone_trainable(model, trainable: bool):
    if MODEL_NAME == "pipeline":
        freeze_prefixes = ("body_", "lh_", "rh_", "face_", "mouth_")
    else:
        freeze_prefixes = ("enc.",)
    for name, param in model.named_parameters():
        if name.startswith(freeze_prefixes):
            param.requires_grad = trainable


def load_pretrained_for_finetune(model, ckpt_path: Path):
    if not ckpt_path or str(ckpt_path) in {"", "."}:
        print("No pretrained checkpoint path configured.")
        return
    if not ckpt_path.exists():
        print(f"Pretrained checkpoint not found: {ckpt_path}")
        return
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = ckpt.get("model_state", ckpt.get("model", ckpt))
    current = model.state_dict()
    loaded, skipped = {}, []
    for key, value in state.items():
        clean_key = key.replace("module.", "")
        if clean_key in current and tuple(current[clean_key].shape) == tuple(value.shape):
            loaded[clean_key] = value
        else:
            skipped.append(clean_key)
    current.update(loaded)
    model.load_state_dict(current, strict=True)
    print(f"Loaded {len(loaded)} tensors from pretrained checkpoint.")
    print(f"Skipped {len(skipped)} tensors, usually classifier/vocab-specific heads.")
    if skipped:
        print("First skipped keys:", skipped[:10])


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_dev_wer, history):
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    torch.save({
        "experiment_name": EXPERIMENT_NAME,
        "model_name": MODEL_NAME,
        "stage": STAGE,
        "dataset_name": DATASET_NAME,
        "epoch": epoch,
        "model_state": raw_model.state_dict(),
        "optimizer_state": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "best_dev_wer": best_dev_wer,
        "history": history,
        "vocab": vocab,
        "config": CONFIG,
        "keypoint_layout": "MediaPipe-86: pose[0:33], left_hand[33:54], right_hand[54:75], face[75:82], mouth[82:86]",
    }, path)


def load_resume_checkpoint(model, optimizer, scheduler):
    resume_path = RESUME_CKPT_PATH
    if not CONFIG.get("resume_training", False) or not resume_path or str(resume_path) in {"", "."}:
        return 0, float("inf"), []
    if not resume_path.exists():
        print(f"Resume checkpoint not found: {resume_path}. Training will start from epoch 0.")
        return 0, float("inf"), []

    ckpt = torch.load(resume_path, map_location="cpu", weights_only=False)
    state = ckpt.get("model_state", ckpt.get("model", ckpt))
    model.load_state_dict(state, strict=True)

    history = ckpt.get("history", []) or []
    best_dev_wer = float(ckpt.get("best_dev_wer", float("inf")))
    start_epoch = int(ckpt.get("epoch", -1)) + 1

    if CONFIG.get("resume_load_optimizer", True):
        try:
            if ckpt.get("optimizer_state") is not None:
                optimizer.load_state_dict(ckpt["optimizer_state"])
            if ckpt.get("scheduler_state") is not None:
                scheduler.load_state_dict(ckpt["scheduler_state"])
            print("Loaded optimizer and scheduler states from resume checkpoint.")
        except Exception as exc:
            print(f"Could not load optimizer/scheduler states cleanly: {exc}")
            print("Continuing with model weights only and a fresh optimizer/scheduler.")

    print(f"Resumed from {resume_path}")
    print(f"Next epoch: {start_epoch}; checkpoint best Dev WER: {best_dev_wer:.4f}")
    del ckpt
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return start_epoch, best_dev_wer, history


def train_model(model, train_loader, dev_loader):
    if STAGE == "finetune":
        load_pretrained_for_finetune(model, PRETRAIN_CKPT_PATH)
        if CONFIG["freeze_backbone_epochs"] > 0:
            set_backbone_trainable(model, False)
            print(f"Backbone frozen for first {CONFIG['freeze_backbone_epochs']} epochs.")

    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer, CONFIG["epochs"])
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available() and CONFIG.get("amp_enabled", True))
    ctc_fn = nn.CTCLoss(blank=0, zero_infinity=True)
    start_epoch, best_dev_wer, history = load_resume_checkpoint(model, optimizer, scheduler)
    patience = 0

    if start_epoch >= CONFIG["epochs"]:
        print(f"Checkpoint is already at epoch {start_epoch - 1}; CONFIG['epochs']={CONFIG['epochs']}. Nothing to train.")
        return history

    for epoch in range(start_epoch, CONFIG["epochs"]):
        if STAGE == "finetune" and epoch == CONFIG["freeze_backbone_epochs"] and CONFIG["freeze_backbone_epochs"] > 0:
            set_backbone_trainable(model, True)
            optimizer = make_optimizer(model)
            scheduler = make_scheduler(optimizer, CONFIG["epochs"] - epoch)
            print("Backbone unfrozen. Optimizer reset for full fine-tuning.")

        model.train()
        running_loss = 0.0
        optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(train_loader, desc=f"epoch {epoch:03d}", leave=False)
        bad_batches = 0
        for step, batch in enumerate(pbar):
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available() and CONFIG.get("amp_enabled", True)):
                loss, _ = compute_training_loss(model, batch, ctc_fn, epoch)
                loss = loss / CONFIG["accum_steps"]
            if CONFIG.get("skip_nonfinite_batches", True) and not torch.isfinite(loss.detach()):
                bad_batches += 1
                optimizer.zero_grad(set_to_none=True)
                print(f"Skipped non-finite loss at epoch {epoch}, step {step}; bad_batches={bad_batches}")
                if bad_batches >= CONFIG.get("max_bad_batches_per_epoch", 3):
                    raise FloatingPointError("Too many non-finite losses in one epoch. Restart from best checkpoint with lower LR.")
                continue
            scaler.scale(loss).backward()
            if (step + 1) % CONFIG["accum_steps"] == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                grad_norm = nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
                if CONFIG.get("skip_nonfinite_batches", True) and not torch.isfinite(grad_norm):
                    bad_batches += 1
                    optimizer.zero_grad(set_to_none=True)
                    scaler.update()
                    print(f"Skipped non-finite gradient at epoch {epoch}, step {step}; bad_batches={bad_batches}")
                    if bad_batches >= CONFIG.get("max_bad_batches_per_epoch", 3):
                        raise FloatingPointError("Too many non-finite gradients in one epoch. Try amp_enabled=False, lower LR, or an earlier checkpoint.")
                    continue
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            running_loss += float(loss.item()) * CONFIG["accum_steps"]
            pbar.set_postfix(loss=f"{running_loss / max(1, step + 1):.4f}")

        scheduler.step()
        train_loss = running_loss / max(1, len(train_loader))
        if not math.isfinite(train_loss):
            raise FloatingPointError("Non-finite epoch loss. Use best_model.pt before this epoch and lower LR.")
        if torch.cuda.is_available() and CONFIG.get("empty_cache_each_epoch", False):
            torch.cuda.empty_cache()
        dev_metrics, _ = evaluate(model, dev_loader, inv_vocab, "dev", decode_method="greedy")
        current_lr = optimizer.param_groups[0]["lr"]
        record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "dev_wer": dev_metrics["wer"],
            "dev_substitution_rate": dev_metrics["substitution_rate"],
            "dev_deletion_rate": dev_metrics["deletion_rate"],
            "dev_insertion_rate": dev_metrics["insertion_rate"],
            "lr": current_lr,
        }
        history.append(record)
        save_csv(history, OUTPUT_DIR / "training_log.csv")
        print(f"Epoch {epoch:03d} | loss={train_loss:.4f} | dev WER={dev_metrics['wer']:.4f} | lr={current_lr:.2e}")

        save_checkpoint(OUTPUT_DIR / "last_model.pt", model, optimizer, scheduler, epoch, best_dev_wer, history)
        if dev_metrics["wer"] < best_dev_wer:
            best_dev_wer = dev_metrics["wer"]
            patience = 0
            save_checkpoint(OUTPUT_DIR / "best_model.pt", model, optimizer, scheduler, epoch, best_dev_wer, history)
            print(f"Saved best model with Dev WER={best_dev_wer:.4f}")
        else:
            patience += 1
            if patience >= CONFIG["patience"]:
                print(f"Early stopping at epoch {epoch}. Best Dev WER={best_dev_wer:.4f}")
                break
        gc.collect()
        if torch.cuda.is_available() and CONFIG.get("empty_cache_each_epoch", False):
            torch.cuda.empty_cache()
    return history


## Plotting and report artifacts


In [6]:
def plot_history(history):
    if not history:
        return
    epochs = [r["epoch"] for r in history]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(epochs, [r["train_loss"] for r in history], marker="o")
    axes[0].set_title("Training CTC Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, [r["dev_wer"] for r in history], marker="o", color="tab:red")
    axes[1].set_title("Dev WER")
    axes[1].set_xlabel("Epoch")
    axes[1].grid(alpha=0.3)

    axes[2].plot(epochs, [r["lr"] for r in history], marker="o", color="tab:green")
    axes[2].set_title("Learning Rate")
    axes[2].set_xlabel("Epoch")
    axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=220)
    plt.close()


def plot_error_breakdown(metrics, name):
    labels = ["Substitution", "Deletion", "Insertion"]
    values = [
        metrics["substitution_rate"] * 100,
        metrics["deletion_rate"] * 100,
        metrics["insertion_rate"] * 100,
    ]
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(labels, values, color=["#4C78A8", "#F58518", "#54A24B"])
    ax.bar_label(bars, fmt="%.1f%%")
    ax.set_ylabel("Rate over reference words (%)")
    ax.set_title(f"{name} WER Breakdown")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{name.lower()}_wer_breakdown.png", dpi=220)
    plt.close()


def aggregate_bucket(rows, key_fn):
    buckets = defaultdict(lambda: Counter())
    for row in rows:
        key = key_fn(row)
        buckets[key]["S"] += int(row["S"])
        buckets[key]["D"] += int(row["D"])
        buckets[key]["I"] += int(row["I"])
        buckets[key]["N"] += int(row["N"])
    out = []
    for key, c in sorted(buckets.items(), key=lambda kv: str(kv[0])):
        wer = (c["S"] + c["D"] + c["I"]) / max(1, c["N"])
        out.append({"bucket": key, "wer": wer, "N": int(c["N"]), "S": int(c["S"]), "D": int(c["D"]), "I": int(c["I"])})
    return out


def length_bucket(row):
    n = int(row["ref_len"])
    if n <= 4:
        return "short_2_4"
    if n <= 7:
        return "medium_5_7"
    return "long_8_plus"


def hand_quality_bucket(row):
    vals = []
    for key in ["lh_detect_rate", "rh_detect_rate"]:
        try:
            vals.append(float(row[key]))
        except Exception:
            pass
    q = min(vals) if vals else 0.0
    if q < 20:
        return "hand_lt_20"
    if q < 40:
        return "hand_20_40"
    if q < 60:
        return "hand_40_60"
    return "hand_60_plus"


def plot_bucket_wer(rows, key_fn, filename, title):
    data = aggregate_bucket(rows, key_fn)
    save_csv(data, OUTPUT_DIR / filename.replace(".png", ".csv"))
    if not data:
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar([d["bucket"] for d in data], [d["wer"] * 100 for d in data], color="#4C78A8")
    ax.bar_label(bars, fmt="%.1f%%")
    ax.set_ylabel("WER (%)")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=220)
    plt.close()


def save_top_error_tables(rows):
    substitutions = Counter()
    deletions = Counter()
    insertions = Counter()
    for row in rows:
        ref = row["reference"].split()
        hyp = row["prediction"].split()
        for op, r, h in edit_ops(ref, hyp)["ops"]:
            if op == "S":
                substitutions[(r, h)] += 1
            elif op == "D":
                deletions[r] += 1
            elif op == "I":
                insertions[h] += 1
    save_csv(
        [{"ref": k[0], "pred": k[1], "count": v} for k, v in substitutions.most_common(50)],
        OUTPUT_DIR / "top_substitutions.csv",
        ["ref", "pred", "count"],
    )
    save_csv(
        [{"ref": k, "count": v} for k, v in deletions.most_common(50)],
        OUTPUT_DIR / "top_deletions.csv",
        ["ref", "count"],
    )
    save_csv(
        [{"pred": k, "count": v} for k, v in insertions.most_common(50)],
        OUTPUT_DIR / "top_insertions.csv",
        ["pred", "count"],
    )


def plot_mska_stream_wers(stream_metrics):
    if not stream_metrics:
        return
    names = list(stream_metrics.keys())
    values = [stream_metrics[n]["wer"] * 100 for n in names]
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(names, values, color=["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#E45756"])
    ax.bar_label(bars, fmt="%.1f%%")
    ax.set_ylabel("WER (%)")
    ax.set_title("MSKA Per-Stream Test WER")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "mska_stream_wer.png", dpi=220)
    plt.close()


## Run experiment

Final evaluation is performed on the test split. The notebook saves prediction CSV files for qualitative inspection.


In [7]:
model = create_model(len(vocab)).to(DEVICE)
num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

def print_model_summary(model):
    summary = {
        "Experiment": EXPERIMENT_NAME,
        "Model": model.__class__.__name__,
        "Stage": STAGE,
        "Dataset": DATASET_NAME,
        "Run mode": RUN_MODE,
        "Vocab size": len(vocab),
        "Input layout": "(T, 86, 3) MediaPipe skeleton",
        "Parameters": f"{num_params:,}",
        "Trainable parameters": f"{trainable_params:,}",
        "Output dir": str(OUTPUT_DIR),
    }
    print("Model summary")
    for key, value in summary.items():
        print(f"  {key:22s}: {value}")


def metric_row(label, metrics):
    return {
        "Metric": label,
        "WER (%)": round(metrics["wer"] * 100, 2),
        "Sub (%)": round(metrics["substitution_rate"] * 100, 2),
        "Del (%)": round(metrics["deletion_rate"] * 100, 2),
        "Ins (%)": round(metrics["insertion_rate"] * 100, 2),
        "S": metrics["S"],
        "D": metrics["D"],
        "I": metrics["I"],
        "N": metrics["N"],
        "Samples": metrics["samples"],
    }


print_model_summary(model)

start_time = time.time()
history = train_model(model, train_loader, dev_loader)
plot_history(history)

best_path = OUTPUT_DIR / "best_model.pt"
best_epoch = None
best_dev_wer = None
if best_path.exists():
    ckpt = torch.load(best_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    best_epoch = ckpt.get("epoch")
    best_dev_wer = ckpt.get("best_dev_wer")
    print(f"Loaded best checkpoint from epoch {ckpt['epoch']} with Dev WER={ckpt['best_dev_wer']:.4f}")
    del ckpt
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

dev_metrics, dev_rows = evaluate(model, dev_loader, inv_vocab, "dev", decode_method="greedy")
test_greedy_metrics, test_greedy_rows = evaluate(model, test_loader, inv_vocab, "test", decode_method="greedy")
if RUN_MODE == "smoke" and not CONFIG["run_beam_in_smoke"]:
    print("Skipping CTC beam search in smoke mode. Greedy decode is enough for sanity check.")
    test_beam_metrics, test_beam_rows = None, []
else:
    test_beam_metrics, test_beam_rows = evaluate(model, test_loader, inv_vocab, "test", decode_method="beam")

save_csv(dev_rows, OUTPUT_DIR / "predictions_dev_greedy.csv")
save_csv(test_greedy_rows, OUTPUT_DIR / "predictions_test_greedy.csv")
if test_beam_rows:
    save_csv(test_beam_rows, OUTPUT_DIR / "predictions_test_beam.csv")

final_metrics = {
    "experiment_name": EXPERIMENT_NAME,
    "model_name": MODEL_NAME,
    "stage": STAGE,
    "dataset_name": DATASET_NAME,
    "run_mode": RUN_MODE,
    "parameters_total": int(num_params),
    "parameters_trainable_initial": int(trainable_params),
    "elapsed_minutes": (time.time() - start_time) / 60.0,
    "best_epoch": best_epoch,
    "best_dev_wer": best_dev_wer,
    "dev_greedy": dev_metrics,
    "test_greedy": test_greedy_metrics,
    "test_beam": test_beam_metrics,
}

if MODEL_NAME == "mska":
    stream_metrics = {}
    for idx, name in enumerate(MSKA_STREAM_NAMES):
        m, _ = evaluate(model, test_loader, inv_vocab, "test", decode_method="greedy", head_index=idx)
        stream_metrics[name] = m
    final_metrics["mska_streams_test_greedy"] = stream_metrics
    save_json(stream_metrics, OUTPUT_DIR / "mska_stream_wers.json")
    plot_mska_stream_wers(stream_metrics)

save_json(final_metrics, OUTPUT_DIR / "metrics.json")
report_metrics = test_beam_metrics if test_beam_metrics is not None else test_greedy_metrics
report_rows = test_beam_rows if test_beam_rows else test_greedy_rows
report_name = "Test_Beam" if test_beam_metrics is not None else "Test_Greedy"
plot_error_breakdown(report_metrics, report_name)
plot_bucket_wer(report_rows, length_bucket, "test_wer_by_sentence_length.png", "Test WER by Sentence Length")
plot_bucket_wer(report_rows, hand_quality_bucket, "test_wer_by_hand_quality.png", "Test WER by Hand Keypoint Quality")
save_top_error_tables(report_rows)

summary_rows = [
    metric_row("Dev Greedy", dev_metrics),
    metric_row("Test Greedy", test_greedy_metrics),
]
if test_beam_metrics is not None:
    summary_rows.append(metric_row("Test Beam", test_beam_metrics))

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "final_metrics_summary.csv", index=False)
print("Final evaluation summary")
display(summary_df)
print("Artifacts saved to:", OUTPUT_DIR)


Model summary
  Experiment            : phoenix14t_mska_plus_pretrain
  Model                 : MSKACSLRModel
  Stage                 : pretrain
  Dataset               : PHOENIX14T
  Run mode              : full
  Vocab size            : 1116
  Input layout          : (T, 86, 3) MediaPipe skeleton
  Parameters            : 28,289,280
  Trainable parameters  : 28,289,280
  Output dir            : /kaggle/working/phoenix14t_mska_plus_pretrain
Loaded optimizer and scheduler states from resume checkpoint.
Resumed from /kaggle/input/datasets/thn1208/phase3/phoenix14t_mska_plus_pretrain/last_model.pt
Next epoch: 122; checkpoint best Dev WER: 0.3797


epoch 122:   0%|          | 0/3548 [00:00<?, ?it/s]

eval dev greedy:   0%|          | 0/519 [00:00<?, ?it/s]

Epoch 122 | loss=0.8449 | dev WER=0.3866 | lr=1.98e-05


epoch 123:   0%|          | 0/3548 [00:00<?, ?it/s]

...

eval test greedy:   0%|          | 0/642 [00:00<?, ?it/s]

eval test greedy:   0%|          | 0/642 [00:00<?, ?it/s]

Final evaluation summary


,Metric,WER (%),Sub (%),Del (%),Ins (%),S,D,I,N,Samples
0,Dev Greedy,38.37,24.01,11.55,2.80,900,433,105,3748,519
1,Test Greedy,37.45,24.18,10.13,3.14,1031,432,134,4264,642
2,Test Beam,37.34,24.53,9.43,3.38,1046,402,144,4264,642


Artifacts saved to: /kaggle/working/phoenix14t_mska_plus_pretrain
